Bibliotecas Necessárias

In [ ]:
# Biblioteca nativa do python que tem como objetivo manipular 
# arquivos json
import json

# Classe da biblioteca bs4 que tem como objetivo coletar dados através
# do HTML das páginas.
from bs4 import BeautifulSoup

# Biblioteca que tem como objetivo enviar requisições a sites e APIs
import requests

# Módulo da biblioteca google que tem como objetivo possibilitar
# a nossa conexão com a API da inteligência artificial da Google
# IA Studio.
from google import genai

# Biblioteca nativa do python que tem como objetivo manipular o sistema 
# de arquivos do sistema operácional 
import os

# Função da biblioteca dotenv que tem como objetivo carregar variáveis de 
# ambientes no código
from dotenv import load_dotenv

# O SMTP (Simple Mail Transfer Protocol) é o protocolo padrão que os 
# servidores usam para enviar mensagens de texto. A classe smtplib
# abre uma conexão segura com o servidor da nossa conta (como o
# smtp.gmail.com), faz o login usando as suas credenciais de segurança
# despacha a caixa (MIMEMultipart) transformada em texto para o
# destinatário.
import smtplib

# É a classe onde colamos as etiquetas externas essenciais, como o remetente
# (From), o destinatário(To) e o assunto (Subject). Dizemos que ela é 
# multipart por que é essa estrutura que permite que o e-mail contenha
# mais de uma coisa ao mesmo tempo lá dentro (como um texto comum, uma
# versão em HTML com cores e um arquivo em anexo). 
from email.mime.multipart import MIMEMultipart

# O python precisa saber se o texto que você escreveu é apenas um texto
# puro e simples ('plain') ou se é um codigo com formatação visual 
# (html). O MIMEText converte a nossa string do python em um formato
# que os leitores de e-mail (como o gmail ou Outlook) consigam interpretar
# corretamente. Após criar esse texto, você o joga dentro do envelope
# MIMEMultipart usando o comando attach() 
from email.mime.text import MIMEText

# Biblioteca que tem como objetivo manipular o tempo.
import time

Acessando a chave da API através das variáveis de ambiente

In [2]:
load_dotenv()

CHAVE_API = os.getenv('CHAVE_API')

EMAIL = os.getenv('EMAIL')

SENHA = os.getenv('SENHA')

Instanciando a classe genai

In [3]:
gemini = genai.Client(api_key=CHAVE_API)

lendo o arquivo de endereços

In [4]:
caminho_enderecos = 'produtos.json'

with open(caminho_enderecos, "r") as enderecos:
    
    dicionario_enderecos = json.load(enderecos)

dicionario_enderecos
    

{'https://www.netshoes.com.br/p/jaqueta-corta-vento-resistente-a-agua-com-refletivos-e-bolso-invisivel-decole-unissex-1ST-0004-042': {'loja': 'netshoes',
  'produto': 'corta vento',
  'preco': 122.0},
 'https://www.kabum.com.br/produto/593862/water-cooler-husky-glacier-iluminacao-argb-120mm-amd-e-intel-preto-hwt500pt': {'loja': 'kabum',
  'produto': 'cooler de PC',
  'preco': 200}}

Inserindo links no sistema (Este trecho só é obrigatório no primeiro acesso)

In [20]:
link = input("Link da página que será acessada")

nome_loja = input("Nome da loja")

nome_produto = input("Nome do produto: ")

faixa_preco = input("Preço alvo (avisar se o valor for menor ou igual a):")


if link != '' and nome_loja != '' and faixa_preco != '':
    
    faixa_preco = faixa_preco.replace(",", ".")

    faixa_preco = float(faixa_preco)
        
    dicionario_enderecos[link] = {"loja": nome_loja, "produto": nome_produto,"preco":faixa_preco}
        
    with open(caminho_enderecos, "w") as arquivo:
        
        json.dump(dicionario_enderecos, arquivo, indent=4)

else:
    
    print("Por favor preencha todos os campos")


dicionario_enderecos

Por favor preencha todos os campos


{'https://www.netshoes.com.br/p/jaqueta-corta-vento-resistente-a-agua-com-refletivos-e-bolso-invisivel-decole-unissex-1ST-0004-042': {'loja': 'netshoes',
  'produto': 'corta vento',
  'preco': 122.0},
 'https://www.kabum.com.br/produto/593862/water-cooler-husky-glacier-iluminacao-argb-120mm-amd-e-intel-preto-hwt500pt': {'loja': 'kabum',
  'produto': 'cooler de PC',
  'preco': 200}}

Configurando o envio de email

In [5]:
def enviar_email(dados_produtos):
    
    try:
        
        servidor_smtp = "smtp.gmail.com"
        
        porta_smtp = 587

        email_remetente = EMAIL

        senha_remetente = SENHA

        email_destinatario = EMAIL

        mensagem = MIMEMultipart()

        mensagem['From'] = email_remetente

        mensagem['To'] = email_destinatario

        mensagem['subject'] = f"Baixa no preço do produto {dados_produtos['produto']}"
        
        texto_mensagem = f"Olá o produto {dados_produtos['produto']} esta abaixo de {dados_produtos['preco']}"
        
        mensagem.attach(MIMEText(texto_mensagem, 'plain'))
        
        servidor = smtplib.SMTP(servidor_smtp, porta_smtp)
        
        servidor.starttls()
        
        servidor.login(email_remetente, senha_remetente)
        
        servidor.sendmail(email_remetente, email_destinatario, mensagem.as_string())
        
        print("O aviso foi enviado para o email do usuário")
    
    except Exception as erro:
        
        print(f"Falha no envio do email: {erro}")
        
        

Acessando a lista de sites

In [6]:
for url, dados_produtos in dicionario_enderecos.items():
    
    time.sleep(60)
    
    headers = {"User-Agent": "Chrome/149.0.7827.103",
               "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
               "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
               "Accept-Encoding": "gzip, deflate, br",
               "Connection": "keep-alive",
               "Upgrade-Insecure-Requests": "1",
               "Sec-Fetch-Dest": "document",
               "Sec-Fetch-Mode": "navigate",
               "Sec-Fetch-Site": "none",
               "Sec-Fetch-User": "?1"}
    
    requisicao = requests.get(url, headers=headers)
    
    if requisicao.status_code != 200:
        
        print(f"Não foi possivel acessar o site {dados_produtos['loja']}. Status: {requisicao.status_code}\n")
    
    else:
        
        html = requisicao.text
        
        sopa = BeautifulSoup(html, "html.parser")
        
        try:
            
        
            resposta = gemini.models.generate_content(
            
            model = 'gemini-2.5-flash',
            contents = f"Análise o HTML de cada url retornada e colete o preço de cada produto. HTML dos e-commerce: {sopa}. Por favor, caso você encontre o preço do produto no html, retorne apenas o valor do produto (sem utilizar qualquer outro texto, apenas o valor bruto), pois irei usar esse valor na automação que estou construindo."
            )
            
            
            if "ERRO" in resposta.text:
                
                print(f"O sistema não encontrou o preço do {dados_produtos['produto']} no site {dados_produtos['loja']}\n")
                
                print(f"{resposta.text}\n")
            
            else:
            
                preco_atual_texto = resposta.text.replace(",",".")
                
                preco_atual = float(preco_atual_texto)
                
                print(f"Loja: {dados_produtos['loja']}")
                print(f"Preço atual: {preco_atual}")
                
                if preco_atual <= dados_produtos['preco']:
                    
                    print(f"A loja {dados_produtos['loja']} esta com o produto {dados_produtos['produto']} abaixo de {dados_produtos['preco']}")
                    
                    enviar_email(dados_produtos)
                    
        except Exception as erro:
            
            print(f"A API do gemini retornou um erro: {erro}")
            
            print(f"Nome da loja que não possui o preço: {dados_produtos['loja']}")

A API do gemini retornou um erro: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 16.76548949s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'l